In [14]:
import os
import matplotlib
import pandas as pd
import xarray as xr
import rioxarray
from itertools import product
from dask.distributed import Client, LocalCluster
import platform
from glob import glob
import numpy as np
import multiprocessing
import dask

Define paths

In [15]:
data_path = 'F:/Wetland_Climate_Impacts/Climate_Data/USGS_Thresholds_Weighted/Grid/temp/all_metrics'
meta_data_file = 'F:/Wetland_Climate_Impacts/Climate_Data/USGS_Water_Balance/NCCV2_LOCA2_model_extended_meta.csv'

Specify weight type

In [16]:
weight_type = 'NCA5_BMA_weight' # or 'LOCA2_BMA_weight'

GHG emission scenarios

In [17]:
scenarios = np.array(['ssp245','ssp370','ssp585'])
n_s = len(scenarios)

Future year intervals

In [18]:
start_dates = np.array(['2041-01-01','2071-01-01'])
end_dates = np.array(['2070-12-31','2100-12-31'])
n_y = len(start_dates)

Temperature variables

In [19]:
variables = np.array(["DTR","FD","ID","SU","TR","TX90p","TX90pDAYS","TX95p","TX95pDAYS",
                      "TXge90F","TXge95F","TXge100F","TXge105F","TXge110F"])

Using xarray's open_mfdataset function, we can easily open and combine all the files at once. However, they need something to align on, so we use a preprocessor function (read the docs) to find the CMIP metadata in each file and construct a ensemble ID (as we did the the metadata DataFrame). We use this ensemble ID to create a new 4th dimension for each model. This allows us to load all models at once.

Some of the variables were loading as float64, which takes more memory and is not needed, so I cast them all to float32 datatype.

In [20]:
def get_ensemble_id_from_netcdf_metadata(ds):
    ensemble = f"{ds.attrs['model_id']}"
    return ds.expand_dims('ensemble').assign_coords(ensemble=[ensemble]).astype('float32')

Define ensemble function

In [22]:
def create_ensemble(scenario, interval_index, variable):      
    # Load the metadata file, filter only for our selected SSP, and create a unqiue ensemble ID (model.scenario.ripf).
    df = pd.read_csv(meta_data_file, encoding="ISO-8859-1")
    df = df[df['EXP'] == scenario]
    df['ensemble'] = df['MODEL']+"."+df['EXP']+"."+df['RIPF']

    # Drop most of the DataFrame and only keep the weights we want and the ensemble ID (also making it an index)
    df_weight = df.set_index('ensemble')[weight_type]

    # Since the ensemble ID is an index, we can easily convert this Pandas DataFrame to a xarray DataArray.
    weight_da = df_weight.to_xarray()
    
    # Use Python expansions to build our list of files
    ssp_files = [
        f'{data_path}/{variable}/CMIP6-LOCA2_Thresholds_{variable}_{ensemble_id}_1950-2100_16thdeg_grid.nc'
        for ensemble_id in list(weight_da.ensemble.values)
    ]

    # Create ensemble for all scenario files
    ds = xr.open_mfdataset(ssp_files, 
                           chunks={}, 
                           preprocess=get_ensemble_id_from_netcdf_metadata, 
                           combine_attrs="drop_conflicts")

    # Add weights to data array
    ds['weights'] = weight_da

    # Average across time slices
    ds_clim_historical = ds.sel(time=slice('1981-01-01','2010-12-31')).mean(dim="time")
    ds_clim_future = ds.sel(time=slice(start_dates[interval_index], end_dates[interval_index])).mean(dim="time")

    # Calculate weighted average across models
    weighted_ds_clim_historical = ds_clim_historical.weighted(ds_clim_historical.weights).mean(dim="ensemble", keep_attrs=True).drop_vars("weights")
    weighted_ds_clim_future = ds_clim_future.weighted(ds_clim_future.weights).mean(dim="ensemble", keep_attrs=True).drop_vars("weights")

    # Calculate difference between future and historical
    weighted_ds_clim_change = weighted_ds_clim_future - weighted_ds_clim_historical
    
    # Save results
    start_year = start_dates[interval_index].split('-')[0]
    end_year = end_dates[interval_index].split('-')[0]
    nc_filename = "F:/Wetland_Climate_Impacts/Climate_Data/USGS_Thresholds_Weighted/Grid/temp/bma_weighted_differences/{}/{}_{}/NCA5_BMA_Weighted_Ensemble_Difference_{}_{}_{}_{}.nc".format(scenario, start_year, end_year, variable, scenario, start_year, end_year)
    weighted_ds_clim_change.rio.set_spatial_dims(x_dim="lon", y_dim="lat")
    weighted_ds_clim_change.rio.write_crs("EPSG:4326", inplace=True) 
    weighted_ds_clim_change.rio.write_crs(4326,inplace=True).rio.set_spatial_dims(
        x_dim="lon",
        y_dim="lat",
        inplace=True
    ).rio.write_coordinate_system(inplace=True)
    weighted_ds_clim_change.to_netcdf(nc_filename)
    
    return nc_filename

In [9]:
hostname = platform.uname()[1]
cluster = LocalCluster() 
client = Client(cluster)
print(f"Dask Dashboard is available at: {client.dashboard_link.replace("127.0.0.1", hostname)}")

C:\Users\Chels\anaconda3\envs\Climate_Ensembles\Lib\site-packages\distributed\node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 59709 instead
  warnings.warn(


Dask Dashboard is available at: http://SpudsCouch:59709/status


In [23]:
for s in scenarios:
    for i in range(n_y):
        for v in variables:
            create_ensemble(s, i, v)

C:\Users\Chels\AppData\Local\Temp\ipykernel_26288\4075725562.py:20: FutureWarning: In a future version, xarray will not decode timedelta values based on the presence of a timedelta-like units attribute by default. Instead it will rely on the presence of a timedelta64 dtype attribute, which is now xarray's default way of encoding timedelta64 values. To continue decoding timedeltas based on the presence of a timedelta-like units attribute, users will need to explicitly opt-in by passing True or CFTimedeltaCoder(decode_via_units=True) to decode_timedelta. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  ds = xr.open_mfdataset(ssp_files,
C:\Users\Chels\AppData\Local\Temp\ipykernel_26288\4075725562.py:20: FutureWarning: In a future version, xarray will not decode timedelta values based on the presence of a timedelta-like units attribute by default. Instead it will rely on the presence of a timedelta64 dtype attribute, which is now xarray's de

In [11]:
client.close()